# charlm on a GPU: a 59M-parameter model that reasons step by step

This notebook runs the whole pipeline of [charlm](https://github.com/Lior-Baruch/Char_Transformer_Language_Model)
on a Colab GPU, with a model ~30x larger than the CPU examples in the README:

1. **Pretrain** a 59M-parameter character-level GPT (12 layers, 640 wide, 512-character context) on
   [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories), ~1 billion characters of simple English stories.
2. **SFT** it into a chat model for the tasks (reverse, uppercase, spell, length, +, -, *, / and word problems such as
   *"Adam has 12 apples to divide equally among 3 friends. How many each?"*). With `reasoning: true` it learns to
   write a step-by-step scratchpad between `<|think|>` and `<|/think|>` before each math answer.
3. **GRPO**: reinforcement learning, rewarding correct final answers.
4. **DPO**: preference tuning on pairs of a correct reply and one of the model's own wrong replies.

GRPO and DPO are two alternative ways to improve the SFT model: each starts from it, and "Run all" trains both
(switch either off in the settings cell below). The word problems come in four phrasings per operation, numbered
0-3. SFT learns phrasings 0 and 1; GRPO and DPO practise phrasing 2, which the SFT model has never seen, so they have
real mistakes to learn from; phrasing 3 is never trained on and measures how well the model reads wording it never
saw. (When the GPU run's SFT model already got 99% of its training problems right, GRPO and DPO had almost nothing
to learn and made it worse; see the README.)

**Runtime:** choose *Runtime > Change runtime type > A100 GPU* (or L4). Training runs in 16-bit precision
(bf16) automatically. One run took about 3 minutes for the data, 1 h 52 min for pretraining and 5 minutes for SFT;
GRPO and DPO should take ~10-15 minutes each with these configs (an estimate: they were shortened after that run).
An L4 is roughly 3x slower: set `MAX_ITERS = 20000` below for a pretraining run of about the same ~2 h.

**Disconnects:** data and checkpoints are kept on your Google Drive (in `MyDrive/charlm`). If Colab disconnects,
run all the cells again: finished stages are skipped and an interrupted one continues from its last saved state.
If a stage fails, the notebook stops there, so the next stage never starts from an unfinished model.

In [ ]:
# settings
BRANCH = 'master'  # the branch of the repository to run
USE_DRIVE = True  # keep data and checkpoints on Google Drive, so a disconnect loses nothing
MAX_ITERS = None  # pretraining steps; None = the config's 60000 (use 20000 on an L4)
# set MAX_ITERS before pretraining starts. To change it afterwards, delete checkpoints/base.* in MyDrive/charlm,
# and the later stages' checkpoints (sft.*, grpo.*, dpo.*) too if they have run, since they build on the old base
TRAIN_GRPO = True  # reinforcement learning after SFT (set False to skip it)
TRAIN_DPO = True  # preference tuning after SFT, from the same SFT model as GRPO (set False to skip it)

In [ ]:
# setup: clone (or update) the repository, install it and link the storage folder. Safe to run again
import os
import subprocess

os.environ['PYTHONUNBUFFERED'] = '1'  # show training progress as it happens


def run(command):
    """ run a shell command, show its output as it comes, and stop the notebook if it fails """
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
    if process.wait():
        raise RuntimeError(f'failed with exit code {process.returncode}: {command}')


REPO = '/content/Char_Transformer_Language_Model'
if not os.path.exists(REPO):
    run(f'git clone --branch {BRANCH} https://github.com/Lior-Baruch/Char_Transformer_Language_Model {REPO}')
else:
    run(f'git -C {REPO} fetch origin {BRANCH} && git -C {REPO} checkout {BRANCH} '
        f'&& git -C {REPO} pull --ff-only origin {BRANCH}')
os.chdir(REPO)
run('pip install -q -e .')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = '/content/drive/MyDrive/charlm'
else:
    STORE = '/content/charlm'
os.makedirs(f'{STORE}/checkpoints', exist_ok=True)
os.makedirs(f'{STORE}/data', exist_ok=True)
if not os.path.lexists('checkpoints/colab'):
    os.symlink(f'{STORE}/checkpoints', 'checkpoints/colab')  # the configs save to checkpoints/colab/
run('nvidia-smi --query-gpu=name,memory.total --format=csv')

## 1. Data

`prepare-data` streams TinyStories from Hugging Face, cleans it to plain ASCII (the tokenizer's characters) and
keeps the first billion characters (~1 GB). It runs once: later runs find the file on Drive and skip it.
The file is then copied to the Colab machine's disk, which reads faster than Drive.

In [ ]:
import shutil
run(f'python -m charlm prepare-data tinystories --out {STORE}/data/tinystories.txt --max-chars 1000000000')
src = f'{STORE}/data/tinystories.txt'
if not os.path.exists('data/tinystories.txt') or os.path.getsize('data/tinystories.txt') != os.path.getsize(src):
    shutil.copy(src, 'data/tinystories.txt')
print(open('data/tinystories.txt').read(600))

## 2. Pretraining

The model learns to predict the next character of the stories. Watch `val_loss` fall from ~4.6 (a random guess
among 100 characters); the checkpoint with the lowest `val_loss` is kept. The resume state is saved every 5 evaluations,
so a disconnect costs at most ~5000 steps.

In [ ]:
extra = f'--set max_iters={MAX_ITERS}' if MAX_ITERS else ''
run(f'python -m charlm pretrain --config configs/colab/pretrain.json {extra}')

In [ ]:
# the base model writes stories, but it can't follow instructions yet
run('python -m charlm generate --model checkpoints/colab/base.pt --prompt "Once upon a time" --max-new-tokens 600 '
    '--temperature 0.8')

## 3. SFT with reasoning

Supervised fine-tuning on synthetic tasks. The math replies look like this (the model writes all of it):

```
<|user|>What is 47 + 85?<|assistant|><|think|>7+5+0=12 A=2, 4+8+1=13 A=132 => 132<|/think|>132<|end|>
```

Each step is one digit, with the carry written out and `A=` holding the answer built so far, so a small model
can learn it. Word problems first write the equation they need (`12/3: ...`). SFT trains on word-problem phrasings
0 and 1, and the evaluation asks them in phrasing 3, which is never trained on.

In [ ]:
run('python -m charlm sft --config configs/colab/sft.json')

In [ ]:
TASKS = 'reverse uppercase spell length add sub mul div word'
run(f'python -m charlm eval --model checkpoints/colab/sft.pt --tasks {TASKS} --n-per-task 200 --show 9')

In [ ]:
for message in ['Adam has 12 apples to divide equally among 3 friends. How many each?', 'What is 704 - 358?',
                'What is 386 * 7?']:
    print('>', message)
    run(f'python -m charlm chat --model checkpoints/colab/sft.pt "{message}"')

## 4. GRPO: reinforcement learning

GRPO practises word problems in phrasing 2, which SFT never saw. The model samples 16 replies to each problem;
replies with a correct final answer are reinforced relative to the others. A reply whose reasoning is cut off never
reaches an answer and gets reward 0. Watch `reward` climb as the model learns the new phrasing from rewards alone
(on the CPU model it went from 31% to 98% within ~50 steps).

In [ ]:
if TRAIN_GRPO:
    run('python -m charlm grpo --config configs/colab/grpo.json')
else:
    print('skipped: TRAIN_GRPO is False')

## 5. DPO: preference tuning

DPO learns from pairs instead of rewards: the correct reasoning (chosen) and one of the SFT model's own wrong replies
(rejected), on word problems in phrasing 2 like GRPO. It first samples replies to find the model's mistakes, then
trains on the pairs (`dpo.pairs.jsonl`).

In [ ]:
if TRAIN_DPO:
    run('python -m charlm dpo --config configs/colab/dpo.json')
else:
    print('skipped: TRAIN_DPO is False')

## 6. Compare the models

Accuracy of the SFT model and of each model trained from it, on 200 held-out prompts per task: first every task
(word problems in the never-trained phrasing 3), then word problems in phrasing 2, the one GRPO and DPO practised.
Expect GRPO and DPO to win on phrasing 2 and not on phrasing 3: they get better at what they practise.

In [ ]:
TASKS = 'reverse uppercase spell length add sub mul div word'
models = ['checkpoints/colab/sft.pt'] + [f'checkpoints/colab/{name}.pt' for name, on in
                                         [('grpo', TRAIN_GRPO), ('dpo', TRAIN_DPO)] if on]
run(f'python -m charlm eval --model {" ".join(models)} --tasks {TASKS} --n-per-task 200')
run(f'python -m charlm eval --model {" ".join(models)} --tasks word --phrasing 2 --n-per-task 200')

## 7. Training curves

In [ ]:
import json
import matplotlib.pyplot as plt


def load(name):
    path = f'checkpoints/colab/{name}.metrics.jsonl'
    return [json.loads(line) for line in open(path)] if os.path.exists(path) else []


stages = [name for name in ['sft', 'grpo', 'dpo'] if load(name)]
fig, axes = plt.subplots(1, 1 + len(stages), figsize=(5 * (1 + len(stages)), 4))
base = load('base')
axes[0].plot([r['step'] for r in base], [r['val_loss'] for r in base], label='val')
axes[0].plot([r['step'] for r in base], [r['train_loss'] for r in base], label='train')
axes[0].set(title='pretraining loss', xlabel='step'); axes[0].legend()
for ax, name in zip(axes[1:], stages):
    rows = load(name)
    for key in sorted({k for r in rows for k in r if k.startswith('acc/')}):
        ax.plot([r['step'] for r in rows if key in r], [r[key] for r in rows if key in r], label=key[4:])
    ax.set(title=f'{name}: accuracy on held-out prompts', xlabel='step', ylim=(0, 1)); ax.legend(fontsize=8)
plt.show()